# Lesson 6: Passive Moderation — Wiring the Agent into `on_message()`

## WHY
Up until now, your moderation agent runs in a notebook when you manually invoke it.
A real moderation bot needs to **passively watch every message** and decide in real-time whether to act.

In Discord (py-cord), the `on_message()` event fires for every message the bot can see.
That's where your agent will live — but going from "works in a notebook" to "works in production" requires solving several problems:

1. **Async** — Discord is async (`async def on_message`), so your agent must use `ainvoke()`, not `invoke()`
2. **Error handling** — A crash in `on_message()` silently kills that event. No error message, no retry. The message just goes unmoderated
3. **Cost awareness** — Running a full agent on every message gets expensive fast. A server with 1,000 messages/day at ~$0.002/agent run = $60/month on a quiet server
4. **Structured logging** — `print()` statements disappear in production. You need `logging` for observability
5. **Ignoring the bot's own messages** — Without this guard, the bot moderates itself in an infinite loop

This lesson builds the patterns you'll copy into the real bot. Everything here is designed to be importable.

**By the end of this notebook you will:**
1. Understand the difference between `invoke()` and `ainvoke()` and why Discord requires async
2. Build the moderation agent using `create_agent` with model strings
3. Implement defensive error handling around agent calls
4. Add structured logging with Python's `logging` module
5. Build a two-stage pipeline (fast triage → full agent) for cost control
6. Package everything as importable functions ready for `bot.py`

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## WHAT — Sync vs Async in LangChain

Every LangChain runnable (chat models, agents, chains) has **two invocation modes**:

| Method | When to use | Blocks the event loop? |
|--------|-------------|------------------------|
| `.invoke()` | Scripts, notebooks, sync code | Yes |
| `.ainvoke()` | Discord handlers, FastAPI, any `async def` | No |

**Critical rule:** Inside an `async def` function (like `on_message()`), you **must** use `ainvoke()`.
Using `invoke()` inside async code blocks the entire event loop — your bot freezes for every message until the LLM responds.

The same split exists for streaming:
- `.stream()` → sync generator
- `.astream()` → async generator

The input/output format is identical — only the calling convention changes.

```python
# In a notebook (sync) — what we've been doing
result = agent.invoke({"messages": [...]})

# In Discord on_message (async) — what we need now
result = await agent.ainvoke({"messages": [...]})
```

## HOW — Building the Moderation Agent (Review)

Let's set up the tools and agent from Lessons 3-5. This is review — the new content starts with async invocation.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.tools import tool


# ── Structured output models (from Lesson 5) ──────────────────

class QuickTriageVerdict(BaseModel):
    """Lightweight triage result for fast message screening."""

    flagged: bool = Field(
        description="True if the message might violate rules and needs full review"
    )
    reason: str = Field(
        description="Brief explanation of why the message was flagged or passed"
    )


class ModerationVerdict(BaseModel):
    """A structured moderation decision for a Discord message."""

    action: Literal["allow", "warn", "mute", "kick", "ban"] = Field(
        description="The moderation action to take on the message"
    )
    reasoning: str = Field(
        description="Step-by-step explanation of why this action was chosen"
    )
    violated_rule: str | None = Field(
        default=None,
        description="Which specific server rule was violated, if any"
    )
    confidence: float = Field(
        description="Confidence in the decision from 0.0 to 1.0"
    )
    duration_minutes: int = Field(
        default=0,
        description="Duration in minutes for temporary actions. 0 for non-temporary."
    )


# ── Simulated tool data ────────────────────────────────────────

SERVER_RULES = {
    "general": ["Be respectful", "Stay on-topic", "No excessive caps"],
    "spam": ["No self-promo", "No repeated messages", "No phishing links"],
    "nsfw": ["No NSFW outside designated channels", "No gore or shock content"],
}

WARN_HISTORY = {
    "user_123": [
        {"reason": "Spam links in #general", "date": "2026-03-15"},
        {"reason": "Harassment", "date": "2026-04-01"},
    ],
    "user_456": [{"reason": "NSFW in wrong channel", "date": "2026-02-20"}],
    "user_789": [],
}


@tool
def get_server_rules(category: str = "all") -> str:
    """Look up the Discord server's moderation rules.

    Parameters
    ----------
    category : str
        Rule category: 'general', 'spam', 'nsfw', or 'all'.
    """
    if category == "all":
        return "\n".join(f"{c}: {', '.join(r)}" for c, r in SERVER_RULES.items())
    rules = SERVER_RULES.get(category)
    if rules:
        return "\n".join(f"{i}. {r}" for i, r in enumerate(rules, 1))
    return f"Unknown category: {category}. Available: {list(SERVER_RULES.keys())}"


class UserLookupInput(BaseModel):
    """Input for user history lookup."""
    user_id: str = Field(description="The Discord user's ID")


@tool(args_schema=UserLookupInput)
def get_user_warn_history(user_id: str) -> str:
    """Look up a user's previous warnings. Check before deciding on action."""
    if user_id not in WARN_HISTORY:
        return f"No record found for user {user_id}"
    warnings = WARN_HISTORY[user_id]
    if not warnings:
        return f"User {user_id} has a clean record (0 warnings)"
    lines = [f"User {user_id}: {len(warnings)} warning(s)"]
    for w in warnings:
        lines.append(f"  - {w['date']}: {w['reason']}")
    return "\n".join(lines)


class ModActionInput(BaseModel):
    """Input for moderation actions."""
    user_id: str = Field(description="Discord user ID")
    action: str = Field(description="Action: 'warn', 'mute', 'kick', or 'ban'")
    reason: str = Field(description="Reason for the action")
    duration_minutes: int = Field(default=0, description="Duration for temp actions. 0 = permanent.")


@tool(args_schema=ModActionInput)
def issue_moderation_action(
    user_id: str, action: str, reason: str, duration_minutes: int = 0
) -> str:
    """Issue a moderation action against a user. Check rules and history FIRST."""
    valid = ["warn", "mute", "kick", "ban"]
    if action not in valid:
        return f"Invalid action. Must be one of: {valid}"
    dur = f" for {duration_minutes}m" if duration_minutes else ""
    return f"[SIM] {action.upper()}{dur} on {user_id}: {reason}"


moderation_tools = [get_server_rules, get_user_warn_history, issue_moderation_action]
print(f"Tools ready: {[t.name for t in moderation_tools]}")

In [ ]:
from langchain.agents import create_agent

MODERATION_SYSTEM_PROMPT = (
    "You are a Discord server moderation bot.\n"
    "WORKFLOW:\n"
    "1. Check the relevant server rules\n"
    "2. Check the user's warning history\n"
    "3. Decide on an appropriate action\n"
    "4. Issue the action if needed\n\n"
    "GUIDELINES:\n"
    "- First offense + minor violation → warn\n"
    "- Repeat offender + minor violation → mute (30 min)\n"
    "- Severe violation (phishing, threats) → ban regardless of history\n"
    "- Always explain your reasoning"
)

# Agent with structured output — uses model string, no ChatOpenAI import needed
moderation_agent = create_agent(
    "openai:gpt-4o-mini",
    tools=moderation_tools,
    system_prompt=MODERATION_SYSTEM_PROMPT,
    response_format=ModerationVerdict,
)

print(f"Agent type: {type(moderation_agent)}")
print("Agent created with model string + structured output")

## HOW — `ainvoke()` (Async Invocation)

In a notebook, we can use `await` directly in code cells (Jupyter handles the event loop for us).
This lets us test async code exactly as it would run inside `on_message()`.

In [ ]:
from langchain_core.messages import HumanMessage

# ainvoke — the async equivalent of invoke
# Same input, same output — just use `await`
result = await moderation_agent.ainvoke({
    "messages": [
        HumanMessage(
            content=(
                "User user_123 posted in #general: "
                "'FREE DISCORD NITRO! Click: totally-not-a-scam.com'\n"
                "Evaluate and take action."
            )
        )
    ]
})

verdict = result["structured_response"]
print(f"Action: {verdict.action}")
print(f"Confidence: {verdict.confidence}")
print(f"Reasoning: {verdict.reasoning}")

That's it — `ainvoke()` is identical to `invoke()` except you `await` it.
The result dict has the same keys: `messages`, `structured_response`, etc.

**In the bot**, the call would look like:
```python
async def on_message(self, message: discord.Message):
    result = await self.moderation_agent.ainvoke({...})
    verdict = result["structured_response"]
```

## WHAT — Structured Logging (Replacing `print()`)

Every lesson so far used `print()`. That's fine for notebooks, but in a running Discord bot:
- `print()` goes to stdout — which you may not be watching
- No timestamps, no severity levels, no filtering
- No way to distinguish "informational" from "error" from "debug"

Python's built-in `logging` module solves all of this. It's not a dependency — it's in the standard library.

```python
import logging
logger = logging.getLogger(__name__)

logger.info("Message moderated")           # normal operation
logger.warning("Agent took 5+ seconds")     # something to watch
logger.error("Agent call failed", exc_info=True)  # exception with traceback
```

📖 [Python logging HOWTO](https://docs.python.org/3/howto/logging.html)

In [ ]:
import logging

# Configure logging for the notebook
# In the real bot, you'd configure this once at startup in bot.py
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

# Create a logger for this module
# __name__ gives "__main__" in notebooks, but in real code it's the module path
logger = logging.getLogger("moderation")

# Demo the different levels
logger.debug("This won't show — level is INFO")  # hidden
logger.info("Bot started, moderation agent ready")
logger.warning("Agent response took 3.2 seconds")
logger.error("Agent invocation failed")

### Log Levels

| Level | When to use | Example |
|-------|-------------|----------|
| `DEBUG` | Verbose dev info | `"Agent trace: 5 messages, 3 tool calls"` |
| `INFO` | Normal operations | `"Moderated message from user_123: allow"` |
| `WARNING` | Something unexpected but not broken | `"Agent took 5.2s (threshold: 3s)"` |
| `ERROR` | Something failed | `"Agent invocation failed for message 12345"` |

In production, you'd typically run at `INFO` level and switch to `DEBUG` when troubleshooting.

## HOW — Error Handling for Agent Calls

Agent calls can fail for many reasons:
- OpenAI API rate limit or outage
- Network timeout
- Tool raises an unhandled exception
- Model returns unparseable structured output

If any of these happen inside `on_message()` without a try/except, the event silently fails.
The message goes unmoderated with **zero indication** that something went wrong.

The fix is simple: wrap every agent call in try/except and log the failure.

In [ ]:
import time


async def run_moderation_agent(
    agent,
    user_id: str,
    message_content: str,
) -> ModerationVerdict | None:
    """Run the moderation agent with error handling and timing.

    Parameters
    ----------
    agent : CompiledStateGraph
        The moderation agent (created via create_agent with response_format).
    user_id : str
        The Discord user ID who sent the message.
    message_content : str
        The message text to evaluate.

    Returns
    -------
    ModerationVerdict | None
        The structured verdict, or None if the agent call failed.
    """
    start = time.monotonic()
    try:
        result = await agent.ainvoke(
            {
                "messages": [
                    HumanMessage(
                        content=(
                            f"User {user_id} posted: '{message_content}'\n"
                            "Evaluate and take action if needed."
                        )
                    )
                ]
            },
            config={"recursion_limit": 15},
        )
        elapsed = time.monotonic() - start
        verdict = result["structured_response"]

        logger.info(
            "Moderation verdict for %s: action=%s confidence=%.2f (%.1fs)",
            user_id,
            verdict.action,
            verdict.confidence,
            elapsed,
        )
        if elapsed > 3.0:
            logger.warning("Agent took %.1fs (threshold: 3s)", elapsed)

        return verdict

    except Exception:
        elapsed = time.monotonic() - start
        logger.error(
            "Agent failed for user %s after %.1fs",
            user_id,
            elapsed,
            exc_info=True,
        )
        return None


print("run_moderation_agent() defined")

In [ ]:
# Test: Successful invocation
verdict = await run_moderation_agent(
    moderation_agent,
    user_id="user_123",
    message_content="FREE DISCORD NITRO! Click: totally-not-a-scam.com",
)

if verdict:
    print(f"\nAction: {verdict.action}")
    print(f"Reasoning: {verdict.reasoning}")
else:
    print("\nAgent failed — check logs above")

In [ ]:
# Test: What happens when the agent fails?
# Create a deliberately broken agent to simulate failure
from langchain_core.tools import tool as tool_decorator


@tool_decorator
def always_fails(query: str) -> str:
    """A tool that always crashes. Use for any query."""
    raise RuntimeError("Simulated API outage!")


broken_agent = create_agent(
    "openai:gpt-4o-mini",
    tools=[always_fails],
    system_prompt="Always use the always_fails tool.",
    response_format=ModerationVerdict,
)

# This should fail gracefully — no crash, just logs + returns None
verdict = await run_moderation_agent(
    broken_agent,
    user_id="user_456",
    message_content="test message",
)

print(f"\nResult: {verdict}")
print("→ None means the failure was caught. The bot keeps running.")

### Key Patterns in the Error Handler

1. **`time.monotonic()`** — Measures wall-clock time. Use `monotonic()` instead of `time()` because it can't go backwards (clock adjustments)
2. **`exc_info=True`** — Tells the logger to include the full traceback. Without this, you'd only see "Agent failed" with no clue why
3. **Return `None` on failure** — The caller checks `if verdict is None` and skips moderation. Fail-open is safer than crashing the bot
4. **Slow-response warning** — Logs a warning if the agent takes too long. Useful for spotting API degradation before it becomes an outage

## HOW — Two-Stage Pipeline with Async

From Lesson 5, you know the cost-saving pattern: cheap triage first, full agent only when flagged.
Let's build the async version.

In [ ]:
from langchain_openai import ChatOpenAI

# Triage model — lightweight, single LLM call
triage_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(
    QuickTriageVerdict
)

TRIAGE_SYSTEM_PROMPT = (
    "You are a fast message triage system.\n"
    "Quickly decide if a message MIGHT violate Discord rules.\n"
    "When in doubt, flag it — a more thorough review will follow.\n"
    "Rules: no spam, no phishing, no harassment, be respectful."
)

print("Triage model ready")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


async def moderate_message(
    user_id: str,
    message_content: str,
) -> ModerationVerdict | None:
    """Two-stage async moderation: fast triage, then full agent if flagged.

    Parameters
    ----------
    user_id : str
        The Discord user ID who sent the message.
    message_content : str
        The message text to evaluate.

    Returns
    -------
    ModerationVerdict | None
        A structured verdict if the message was flagged and reviewed,
        None if the message passed triage or an error occurred.
    """
    # Stage 1: Fast triage
    try:
        triage = await triage_llm.ainvoke([
            SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
            HumanMessage(content=f"User {user_id} posted: '{message_content}'"),
        ])
        logger.info(
            "Triage for %s: flagged=%s reason=%s",
            user_id,
            triage.flagged,
            triage.reason,
        )
    except Exception:
        logger.error("Triage failed for user %s", user_id, exc_info=True)
        return None

    if not triage.flagged:
        return None  # safe message, skip full agent

    # Stage 2: Full agent (only for flagged messages)
    logger.info("Message from %s flagged — running full agent", user_id)
    return await run_moderation_agent(
        moderation_agent,
        user_id=user_id,
        message_content=message_content,
    )


print("moderate_message() defined — async two-stage pipeline")

In [ ]:
# Test: Harmless message — should stop at triage
result1 = await moderate_message("user_789", "Hey does anyone know a good Python tutorial?")
print(f"Result: {result1}")
print("→ None = passed triage, no agent cost incurred")

In [ ]:
# Test: Suspicious message — should trigger full agent
result2 = await moderate_message("user_123", "FREE DISCORD NITRO! Click: totally-not-a-scam.com")
if result2:
    print(f"\nAction: {result2.action}")
    print(f"Confidence: {result2.confidence}")
    print(f"Reasoning: {result2.reasoning}")

## WHAT — Cost Awareness

Every LLM call costs money. Here's how to think about it:

### Token-Based Pricing (GPT-4o-mini, April 2026)

| Operation | Input tokens | Output tokens | Approx cost per call |
|-----------|-------------|---------------|---------------------|
| Triage (1 call) | ~200 | ~50 | ~$0.0001 |
| Full agent (3-5 calls) | ~2000 | ~500 | ~$0.002 |

### Daily Cost Estimates

| Server size | Messages/day | If 10% flagged | Daily cost |
|-------------|-------------|----------------|------------|
| Small (50 users) | 500 | 50 full agent runs | ~$0.15 |
| Medium (500 users) | 5,000 | 500 full agent runs | ~$1.50 |
| Large (5,000 users) | 50,000 | 5,000 full agent runs | ~$15.00 |

Without triage (running full agent on every message), multiply costs by **10x**.

### Cost Controls
1. **Two-stage pipeline** — triage filters 90%+ of messages (already implemented)
2. **Rate limiting** — cap agent calls per user/minute to prevent abuse
3. **Message length filter** — skip very short messages ("hi", "lol") before even triaging
4. **Caching** — identical messages get cached results (covered in Module 8, Lesson 14)

In [ ]:
# Simple pre-filter: skip messages that are too short to be violations
MIN_MESSAGE_LENGTH = 5  # "hi" and "lol" don't need moderation


async def should_moderate(user_id: str, message_content: str, bot_user_id: str) -> bool:
    """Quick checks before spending tokens on moderation.

    Parameters
    ----------
    user_id : str
        The message author's ID.
    message_content : str
        The message text.
    bot_user_id : str
        The bot's own user ID (to avoid self-moderation).

    Returns
    -------
    bool
        True if the message should be sent to moderation.
    """
    # Don't moderate the bot's own messages
    if user_id == bot_user_id:
        return False

    # Skip very short messages
    if len(message_content.strip()) < MIN_MESSAGE_LENGTH:
        return False

    # Skip empty messages (image-only, embed-only)
    if not message_content.strip():
        return False

    return True


# Test the filter
print(await should_moderate("user_123", "hi", "bot_001"))          # False: too short
print(await should_moderate("bot_001", "spam spam", "bot_001"))    # False: bot's own
print(await should_moderate("user_123", "Check this link!", "bot_001"))  # True

## HOW — The Complete `on_message()` Handler

Here's how all the pieces fit together in a Discord `on_message()` event.
This is a simulation — we can't run a real Discord bot in a notebook — but the code structure is exactly what you'd paste into `AgentBot`.

In [ ]:
async def on_message_handler(
    user_id: str,
    username: str,
    message_content: str,
    bot_user_id: str,
) -> None:
    """Simulated on_message handler — mirrors what goes into AgentBot.

    Parameters
    ----------
    user_id : str
        The message author's user ID.
    username : str
        The message author's display name (for logging).
    message_content : str
        The message text.
    bot_user_id : str
        The bot's own user ID.
    """
    # Step 1: Quick filters (free — no LLM cost)
    if not await should_moderate(user_id, message_content, bot_user_id):
        logger.debug("Skipping message from %s (filtered)", username)
        return

    # Step 2: Two-stage moderation (triage → full agent if flagged)
    verdict = await moderate_message(user_id, message_content)

    # Step 3: Act on the verdict
    if verdict is None:
        # Either passed triage or agent failed — either way, no action
        return

    if verdict.action == "allow":
        logger.info("Message from %s allowed: %s", username, verdict.reasoning)
        return

    # Non-allow actions — this is where you'd call Discord API
    logger.warning(
        "MODERATION ACTION on %s: %s (confidence: %.2f) — %s",
        username,
        verdict.action,
        verdict.confidence,
        verdict.reasoning,
    )

    # In real bot code, you'd do:
    # if verdict.action == "mute":
    #     await message.author.timeout_for(
    #         duration=timedelta(minutes=verdict.duration_minutes),
    #         reason=verdict.reasoning
    #     )
    # if verdict.action == "ban":
    #     await message.author.ban(reason=verdict.reasoning)


print("on_message_handler() defined")

In [ ]:
# Simulate several messages flowing through the handler
print("=" * 60)
print("Simulating message flow...")
print("=" * 60)

messages = [
    ("user_789", "Alice", "hi"),                                    # filtered: too short
    ("bot_001", "ModBot", "I just banned someone"),                 # filtered: bot's own
    ("user_789", "Alice", "Can someone help me with Python?"),      # triage: pass
    ("user_123", "Bob", "FREE NITRO! Click: scam-link.com"),       # triage: flag → agent
]

for uid, name, content in messages:
    print(f"\n--- Message from {name}: '{content}' ---")
    await on_message_handler(uid, name, content, bot_user_id="bot_001")

## Deep Dive — How This Maps to the Real Bot

Here's how the notebook code maps to the actual `AgentBot` class:

| Notebook | Bot code |
|----------|----------|
| `moderation_agent = create_agent(...)` | Created in `AgentBot.__init__()` |
| `triage_llm = ChatOpenAI(...).with_structured_output(...)` | Created in `AgentBot.__init__()` |
| `on_message_handler(user_id, ...)` | `async def on_message(self, message: discord.Message)` |
| `user_id` / `message_content` params | Extracted from `message.author.id` / `message.content` |
| `logger.warning("MODERATION ACTION...")` | Replaced with actual `message.author.timeout_for()` / `.ban()` etc. |

The business logic (triage → agent → act on verdict) is identical. Only the Discord wiring differs.

## HOW — Async Streaming (Optional)

For longer moderation decisions, you might want to stream the agent's reasoning in real-time.
`astream()` is the async version of `stream()`.

In [ ]:
# astream — see each agent step as it happens
async for step in moderation_agent.astream({
    "messages": [
        HumanMessage(content="User user_456 said: 'You're all garbage'. Evaluate.")
    ]
}):
    for node_name, node_output in step.items():
        if node_name == "__end__":
            continue
        print(f"--- Node: {node_name} ---")
        for msg in node_output.get("messages", []):
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  Tool call: {tc['name']}({tc['args']})")
            elif msg.__class__.__name__ == "ToolMessage":
                print(f"  Tool result: {msg.content[:80]}")
            elif msg.content:
                print(f"  {msg.content[:120]}")

## Importable Code

Here's the complete moderation handler packaged for import into the bot.

In [ ]:
"""Moderation handler — copy into src/agentic_discord_moderation_bot/utils/moderation.py"""

import logging
import time
from typing import Literal

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import BaseTool
from langchain_openai import ChatOpenAI
from langgraph.graph.state import CompiledStateGraph
from pydantic import BaseModel, Field

logger = logging.getLogger(__name__)

MIN_MESSAGE_LENGTH = 5


# ── Structured output models ──────────────────────────────────

class QuickTriageVerdict(BaseModel):
    """Lightweight triage result for fast message screening."""

    flagged: bool = Field(
        description="True if the message might violate rules and needs full review"
    )
    reason: str = Field(
        description="Brief explanation of why the message was flagged or passed"
    )


class ModerationVerdict(BaseModel):
    """A structured moderation decision for a Discord message."""

    action: Literal["allow", "warn", "mute", "kick", "ban"] = Field(
        description="The moderation action to take on the message"
    )
    reasoning: str = Field(
        description="Step-by-step explanation of why this action was chosen"
    )
    violated_rule: str | None = Field(
        default=None,
        description="Which specific server rule was violated, if any"
    )
    confidence: float = Field(
        description="Confidence in the decision from 0.0 to 1.0"
    )
    duration_minutes: int = Field(
        default=0,
        description="Duration in minutes for temporary actions. 0 for non-temporary."
    )


# ── Agent factory ─────────────────────────────────────────────

DEFAULT_MODERATION_PROMPT = (
    "You are a Discord server moderation bot.\n"
    "WORKFLOW:\n"
    "1. Check the relevant server rules\n"
    "2. Check the user's warning history\n"
    "3. Decide on an appropriate action\n"
    "4. Issue the action if needed\n\n"
    "GUIDELINES:\n"
    "- First offense + minor violation → warn\n"
    "- Repeat offender + minor violation → mute (30 min)\n"
    "- Severe violation (phishing, threats) → ban regardless of history\n"
    "- Always explain your reasoning"
)

TRIAGE_SYSTEM_PROMPT = (
    "You are a fast message triage system.\n"
    "Quickly decide if a message MIGHT violate Discord rules.\n"
    "When in doubt, flag it — a more thorough review will follow.\n"
    "Rules: no spam, no phishing, no harassment, be respectful."
)


def create_moderation_agent(
    tools: list[BaseTool],
    model: str = "openai:gpt-4o-mini",
    system_prompt: str = DEFAULT_MODERATION_PROMPT,
) -> CompiledStateGraph:
    """Create a moderation agent with structured output.

    Parameters
    ----------
    tools : list[BaseTool]
        Moderation tools the agent can use.
    model : str
        Model string identifier.
    system_prompt : str
        System instructions for the agent.

    Returns
    -------
    CompiledStateGraph
        A ready-to-use moderation agent.
    """
    return create_agent(
        model,
        tools=tools,
        system_prompt=system_prompt,
        response_format=ModerationVerdict,
    )


def create_triage_model(
    model: str = "gpt-4o-mini",
    temperature: float = 0,
):
    """Create the lightweight triage classifier.

    Parameters
    ----------
    model : str
        OpenAI model name.
    temperature : float
        Sampling temperature.

    Returns
    -------
    Runnable
        A structured-output model that returns QuickTriageVerdict.
    """
    return ChatOpenAI(model=model, temperature=temperature).with_structured_output(
        QuickTriageVerdict
    )


# ── Moderation pipeline ───────────────────────────────────────

async def run_moderation_agent(
    agent: CompiledStateGraph,
    user_id: str,
    message_content: str,
) -> ModerationVerdict | None:
    """Run the full moderation agent with error handling.

    Parameters
    ----------
    agent : CompiledStateGraph
        The moderation agent.
    user_id : str
        The Discord user ID.
    message_content : str
        The message text.

    Returns
    -------
    ModerationVerdict | None
        The verdict, or None on failure.
    """
    start = time.monotonic()
    try:
        result = await agent.ainvoke(
            {"messages": [HumanMessage(content=(
                f"User {user_id} posted: '{message_content}'\n"
                "Evaluate and take action if needed."
            ))]},
            config={"recursion_limit": 15},
        )
        elapsed = time.monotonic() - start
        verdict = result["structured_response"]
        logger.info(
            "Moderation verdict for %s: action=%s confidence=%.2f (%.1fs)",
            user_id, verdict.action, verdict.confidence, elapsed,
        )
        return verdict
    except Exception:
        logger.error("Agent failed for user %s", user_id, exc_info=True)
        return None


async def moderate_message(
    triage_model,
    agent: CompiledStateGraph,
    user_id: str,
    message_content: str,
) -> ModerationVerdict | None:
    """Two-stage moderation: triage then full agent if flagged.

    Parameters
    ----------
    triage_model
        The structured-output triage model.
    agent : CompiledStateGraph
        The full moderation agent.
    user_id : str
        The Discord user ID.
    message_content : str
        The message text.

    Returns
    -------
    ModerationVerdict | None
        Verdict if flagged, None if passed or error.
    """
    try:
        triage = await triage_model.ainvoke([
            SystemMessage(content=TRIAGE_SYSTEM_PROMPT),
            HumanMessage(content=f"User {user_id} posted: '{message_content}'"),
        ])
        logger.info("Triage for %s: flagged=%s", user_id, triage.flagged)
    except Exception:
        logger.error("Triage failed for user %s", user_id, exc_info=True)
        return None

    if not triage.flagged:
        return None

    return await run_moderation_agent(agent, user_id, message_content)

## Summary

| Concept | Key takeaway |
|---------|-------------|
| `ainvoke()` / `astream()` | Async equivalents of `invoke()` / `stream()` — required inside `async def` handlers |
| `logging` module | Replace `print()` with `logger.info()`, `.warning()`, `.error()` — gives timestamps, levels, and tracebacks |
| `exc_info=True` | Include full traceback in error logs |
| Error handling | Wrap every agent call in try/except; return `None` on failure (fail-open) |
| `time.monotonic()` | Measure agent latency; warn when above threshold |
| Two-stage pipeline | Triage (cheap) → full agent (expensive) — saves ~80% of LLM cost |
| Pre-filters | Skip short/empty/bot messages before any LLM call (free) |
| Cost awareness | Track token usage; estimate monthly costs before deploying |

**Next up → Lesson 7:** Testing AI systems — use `FakeListChatModel` and `pytest-asyncio` to test your moderation pipeline without spending a cent on API calls.